# CareLoop Clinics – No-Show Guard
## Part 1 — Python & Data Foundations: ingest, validate, clean, enrich

**Inputs:** `data/appointments.csv`, `data/clinics.csv` (originals, never modified)
**Outputs:** `rejected_rows.json`, `appointments_clean.csv`, `clinics_enriched.csv`, `load_data.sql`

In [ ]:
# --- Setup: works in Google Colab (uses Google Drive) and locally ---
import os

try:
    from google.colab import drive          # only exists inside Colab
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CareLoop-NoShow-Guard"
    os.makedirs(os.path.join(PROJECT_DIR, "data"), exist_ok=True)
    os.chdir(PROJECT_DIR)                    # all relative paths start here
except ImportError:
    pass                                     # running locally: use the current folder

print("Working directory:", os.getcwd())

In [ ]:
# Download the three original files into data/ only if they are not there yet.
# Writing raw bytes ("wb") keeps an exact, unmodified copy of each original.
import requests

DATA_URLS = {
    "appointments.csv": "https://coding-platform.s3.amazonaws.com/dev/lms/tickets/c1a9c6a2-5967-41fb-9133-cb6f78008699/qDCCmpyfss1kBJJB.csv",
    "clinics.csv": "https://coding-platform.s3.amazonaws.com/dev/lms/tickets/0fe7e34d-9470-47d8-b7b1-d54b77e009fe/jbFzPdd2gB2Ex2TY.csv",
    "followup_notes.csv": "https://coding-platform.s3.amazonaws.com/dev/lms/tickets/5e122459-7975-4534-957f-8397b0f9c03e/ecWvPsdV2FNmRZAa.csv",
}

for filename, url in DATA_URLS.items():
    path = os.path.join("data", filename)
    if os.path.exists(path):
        print(f"{filename}: already present, not overwritten")
        continue
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(path, "wb") as f:
        f.write(response.content)
    print(f"{filename}: downloaded ({len(response.content):,} bytes)")

## Task 1.1 — Read the file the foundations way (`csv` module only)

`csv.DictReader` returns every value as **text**. `parse_row` converts the numeric columns; if a value
cannot be converted (e.g. a blank `''`), Python raises `ValueError` and the row goes to `rejected`.

In [ ]:
import csv
import json
from collections import Counter

with open("data/appointments.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    headers = reader.fieldnames
    raw_rows = list(reader)

print("Header:", headers)
print("Number of columns:", len(headers))
print("Row count (excluding header):", len(raw_rows))

In [ ]:
INT_COLUMNS = ["age", "fee_inr", "prior_no_shows"]
FLOAT_COLUMNS = ["distance_km"]

def parse_row(row):
    """Return a converted copy of one CSV row.
    Raises ValueError (naming the column) if a value cannot be converted."""
    parsed = dict(row)                       # copy: the original row stays as read
    for col in INT_COLUMNS:
        try:
            parsed[col] = int(row[col])
        except ValueError:
            raise ValueError(f"{col}: cannot convert {row[col]!r} to int")
    for col in FLOAT_COLUMNS:
        try:
            parsed[col] = float(row[col])
        except ValueError:
            raise ValueError(f"{col}: cannot convert {row[col]!r} to float")
    return parsed

accepted = []
rejected = []
for line_number, row in enumerate(raw_rows, start=2):   # line 1 is the header
    try:
        accepted.append(parse_row(row))
    except ValueError as error:
        rejected.append({"line_number": line_number, "error": str(error), "row": row})

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))
print("Rejections by column:", dict(Counter(r["error"].split(":")[0] for r in rejected)))

with open("rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)
print("Saved rejected_rows.json")

In [ ]:
# Checks for Task 1.1
assert len(accepted) + len(rejected) == len(raw_rows)      # no row lost
impossible_ages = [r["age"] for r in accepted if r["age"] < 1 or r["age"] > 100]
print("Accepted rows with impossible ages:", impossible_ages)
print("-> a type check is not a validity check; these are handled in Task 1.3.")

## Task 1.2 — Inspect and clean, pass 1 (pandas)

Starts again from the **original** CSV (Task 1.1 was a separate reading exercise).

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/appointments.csv")

print("Shape:", df.shape)
print()
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())

In [ ]:
print("Rows at start:", len(df))

# (a) drop exact duplicate rows
df = df.drop_duplicates()
print("(a) after dropping duplicates:", len(df))

# (b) clean booking_channel labels
print("\nRaw booking_channel spellings:", df["booking_channel"].nunique(), sorted(df["booking_channel"].dropna().unique()))
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Cleaned booking_channel values:", df["booking_channel"].nunique(), sorted(df["booking_channel"].unique()))
print("(b) after cleaning booking_channel:", len(df))

# (c) convert dates; errors="coerce" turns unreadable appt_date text into NaT (missing)
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("\nMissing appt_date after conversion:", df["appt_date"].isna().sum())
print("Missing status:", df["status"].isna().sum())
print("(c) after converting dates:", len(df))

# (d) drop rows with no usable appointment date or no status
df = df.dropna(subset=["appt_date", "status"])
print("(d) after dropping missing appt_date / status:", len(df))

## Task 1.3 — Clean, pass 2: validity, gaps and outliers

Order matters: invalid ages → fill gaps → **then** quartiles/fences → round.

In [ ]:
# (a) impossible ages become missing
invalid_age = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages:", invalid_age.sum(), "->", df.loc[invalid_age, "age"].tolist())
df.loc[invalid_age, "age"] = np.nan
print("Missing ages now (blank + invalid):", df["age"].isna().sum())

# (b) fill gaps with medians
age_median = df["age"].median()
distance_median = df["distance_km"].median()
print("Median age:", age_median, "| Median distance_km:", distance_median)
df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)

In [ ]:
# (c) IQR fences and capping with np.clip
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_high = (df[col] > upper).sum()
    n_low = (df[col] < lower).sum()
    df[col] = np.clip(df[col], lower, upper)
    print(f"{col}: Q1={q1}, Q3={q3}, IQR={iqr}, lower fence={lower}, upper fence={upper}")
    print(f"   capped {n_high + n_low} values ({n_high} high, {n_low} low)")

# (d) rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("\nLargest fee after capping:", df["fee_inr"].max())
print("Largest distance after capping:", df["distance_km"].max())

## Task 1.4 — Features and a NumPy summary (no loops)

In [ ]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())
print(f"No-shows: {df['no_show'].sum()} of {len(df)} ({df['no_show'].mean() * 100:.1f}%)")

In [ ]:
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) mean of each column
col_means = X.mean(axis=0)
print("Column means [age, distance_km, fee_inr, lead_days]:", np.round(col_means, 2))

# (b) boolean mask: lead_days > 14
lead = X[:, 3]
mask = lead > 14
print(f"Rows with lead_days > 14: {mask.sum()} ({mask.mean() * 100:.1f}%)")

# (c) z-scores of distance_km (np.std = population standard deviation)
x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print("Rows with |z| > 2 for distance_km:", (np.abs(z) > 2).sum())

## Task 1.5 — Enrich the clinics from a live API (Open-Meteo)

`temp_c` = the first hourly `temperature_2m` value (the midnight-UTC reading — a stand-in, not current weather).
Each clinic gets up to 2 attempts; if both fail, `temp_c` stays `None` and the reason is printed. No value is ever invented.

In [ ]:
clinics = pd.read_csv("data/clinics.csv")
URL = "https://api.open-meteo.com/v1/forecast"

def fetch_temperature(lat, lon):
    """Return (temperature, None) on success or (None, reason) on failure."""
    params = {"latitude": lat, "longitude": lon, "hourly": "temperature_2m", "forecast_days": 1}
    try:
        response = requests.get(URL, params=params, timeout=10)
    except requests.exceptions.RequestException as e:
        return None, f"request error: {e}"
    if response.status_code != 200:
        return None, f"status {response.status_code}"
    data = response.json()
    temps = data.get("hourly", {}).get("temperature_2m")
    if not temps or temps[0] is None:
        return None, "hourly temperature_2m missing in reply"
    return temps[0], None

temperatures = []
for _, clinic in clinics.iterrows():
    temp, reason = fetch_temperature(clinic["latitude"], clinic["longitude"])
    if temp is None:                                   # retry once
        temp, reason = fetch_temperature(clinic["latitude"], clinic["longitude"])
    if temp is None:
        print(f"FAILED: {clinic['clinic_name']} -> {reason} (temp_c stored as None)")
    else:
        print(f"OK: {clinic['clinic_name']}: {temp} °C")
    temperatures.append(temp)

clinics["temp_c"] = temperatures
clinics.to_csv("clinics_enriched.csv", index=False)
print("\nSaved clinics_enriched.csv:", clinics.shape)
clinics

## Task 1.6 — Export the clean table and write the SQL load script

In [ ]:
FINAL_COLUMNS = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age", "gender",
                 "distance_km", "fee_inr", "booking_channel", "visit_type", "reminder_sent",
                 "prior_no_shows", "status", "lead_days", "no_show"]

clean = df.sort_values("appointment_id").copy()
clean["booked_on"] = clean["booked_on"].dt.strftime("%Y-%m-%d")
clean["appt_date"] = clean["appt_date"].dt.strftime("%Y-%m-%d")
clean = clean[FINAL_COLUMNS]

# Checks before exporting
print("appointment_id unique:", clean["appointment_id"].is_unique, "| distinct ids:", clean["appointment_id"].nunique())
print("Missing values across the 16 columns:", int(clean.isnull().sum().sum()))
assert clean["appointment_id"].is_unique
assert clean.isnull().sum().sum() == 0

clean.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv:", clean.shape)

In [ ]:
TEXT_APPT = {"appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "gender",
             "booking_channel", "visit_type", "reminder_sent", "status"}
TEXT_CLINIC = {"clinic_id", "clinic_name", "city", "tier"}

def sql_value(value, is_text):
    """Text in single quotes, numbers unquoted, missing values as NULL."""
    if pd.isna(value):
        return "NULL"
    return f"'{value}'" if is_text else f"{value}"

clinics_out = pd.read_csv("clinics_enriched.csv")
appts_out = pd.read_csv("appointments_clean.csv")

lines = []
for _, row in clinics_out.iterrows():           # clinics first (foreign key target)
    values = ", ".join(sql_value(row[c], c in TEXT_CLINIC) for c in clinics_out.columns)
    lines.append(f"INSERT INTO clinics VALUES ({values});")
for _, row in appts_out.iterrows():
    values = ", ".join(sql_value(row[c], c in TEXT_APPT) for c in appts_out.columns)
    lines.append(f"INSERT INTO appointments VALUES ({values});")

with open("load_data.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")

print("INSERT statements written:", len(lines))
print(lines[0])
print(lines[8])

In [ ]:
# Final check of the Part 1 outputs
check = pd.read_csv("appointments_clean.csv")
print("appointments_clean.csv:", check.shape)
print("Columns in order:", list(check.columns) == FINAL_COLUMNS)
with open("load_data.sql", encoding="utf-8") as f:
    sql_lines = f.read().splitlines()
print("INSERT INTO clinics:", sum(l.startswith("INSERT INTO clinics") for l in sql_lines))
print("INSERT INTO appointments:", sum(l.startswith("INSERT INTO appointments") for l in sql_lines))
print("'None' text in SQL file:", any("'None'" in l or "nan" in l for l in sql_lines))